In [6]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset with 12 features per frame --------
class RelativeSpeedDataset240D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            if len(seq) < 15 or sid not in self.distances:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            strdeg = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)

            keys = sorted(self.distances[sid].keys())
            if len(keys) < 15:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            own1 = np.gradient(own)
            own2 = np.gradient(own1)
            str1 = np.gradient(strdeg)
            dist1 = np.gradient(dist)
            dist2 = np.gradient(dist1)

            def smooth(x, w):
                if len(x) < w:
                    return np.zeros_like(x)
                return np.convolve(x, np.ones(w)/w, mode='same')

            sm3 = smooth(dist, 3)
            sm5 = smooth(dist, 5)
            sm7 = smooth(dist, 7)

            for i in range(len(seq) - 14):
                if max_items and len(self.items) >= max_items:
                    return

                rel_speed = tgt[i:i+15] - own[i:i+15]
                if rel_speed.shape[0] != 15:
                    continue

                try:
                    feat = np.stack([
                        own[i:i+15], strdeg[i:i+15], dist[i:i+15],
                        own1[i:i+15], str1[i:i+15], dist1[i:i+15],
                        own2[i:i+15], dist2[i:i+15],
                        sm3[i:i+15], sm5[i:i+15], sm7[i:i+15],
                        rel_speed
                    ], axis=1)  # shape (15, 12)
                except ValueError:
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32), sid

# -------- Model --------
class ExtendedLSTMWithAttention(nn.Module):
    def __init__(self, input_size=12, hidden_size=128):
        super().__init__()
        self.pre_fc = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=hidden_size,
                            num_layers=2, batch_first=True, dropout=0.3)
        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.dropout = nn.Dropout(0.3)
        self.out_fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, T, F = x.size()
        x = self.pre_fc(x.view(-1, F)).view(B, T, -1)
        lstm_out, _ = self.lstm(x)
        attn_scores = self.attn_fc(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.out_fc(self.dropout(context)).squeeze(1)

# -------- Training Function --------
def train_extended_lstm_with_240_features(dataset, save_path="model_extended_240d.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    def collate_fn(batch):
        feats, tgts, sids = zip(*batch)
        return torch.stack(feats), torch.tensor(tgts), list(sids)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ExtendedLSTMWithAttention(input_size=12).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 20
    counter = 0

    for epoch in range(100):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step()

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- Main Block --------
if __name__ == "__main__":
    # データセット作成（最大7500件）
    dataset = RelativeSpeedDataset240D(
        annot_root="../train/train_annotations",
        distance_json_path="../distance3/distance_estimates_corrected.json",
        max_items=7500
    )

    # モデルの学習と保存
    model = train_extended_lstm_with_240_features(
        dataset,
        save_path="model_extended_240d.pth"
    )

    print("✅ 学習完了: model_extended_240d.pth に保存しました")


[Train 1]: 100%|██████████| 93/93 [00:01<00:00, 85.38it/s] 


Epoch 1 | Train Loss: 1.9161 | Val Loss: 0.1599
✅ Saved model to model_extended_240d.pth (val_loss=0.1599)


[Train 2]: 100%|██████████| 93/93 [00:00<00:00, 193.44it/s]


Epoch 2 | Train Loss: 0.2820 | Val Loss: 0.0897
✅ Saved model to model_extended_240d.pth (val_loss=0.0897)


[Train 3]: 100%|██████████| 93/93 [00:00<00:00, 196.02it/s]


...

Epoch 45 | Train Loss: 0.0792 | Val Loss: 0.0552


[Train 46]: 100%|██████████| 93/93 [00:00<00:00, 195.34it/s]


Epoch 46 | Train Loss: 0.0747 | Val Loss: 0.0076


[Train 47]: 100%|██████████| 93/93 [00:00<00:00, 190.45it/s]


Epoch 47 | Train Loss: 0.0651 | Val Loss: 0.0096
🛑 Early stopping at epoch 47
✅ 学習完了: model_extended_240d.pth に保存しました


In [ ]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from model_def import ExtendedLSTMWithAttention  # モデル定義を別ファイルにしている場合
# または同じファイルで定義済みならこの行は不要

# -------- 推論用Dataset（TgtSpeedは不要）--------
class InferenceDataset(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            if len(seq) < 15 or sid not in self.distances:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            strdeg = np.array([f['StrDeg'] for f in seq], dtype=np.float32)

            keys = sorted(self.distances[sid].keys())
            if len(keys) < 15:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            own1 = np.gradient(own)
            own2 = np.gradient(own1)
            str1 = np.gradient(strdeg)
            dist1 = np.gradient(dist)
            dist2 = np.gradient(dist1)

            def smooth(x, w):
                if len(x) < w:
                    return np.zeros_like(x)
                return np.convolve(x, np.ones(w)/w, mode='same')

            sm3 = smooth(dist, 3)
            sm5 = smooth(dist, 5)
            sm7 = smooth(dist, 7)

            for i in range(len(seq) - 14):
                rel_speed = np.zeros(15)
                try:
                    feat = np.stack([
                        own[i:i+15], strdeg[i:i+15], dist[i:i+15],
                        own1[i:i+15], str1[i:i+15], dist1[i:i+15],
                        own2[i:i+15], dist2[i:i+15],
                        sm3[i:i+15], sm5[i:i+15], sm7[i:i+15],
                        rel_speed  # dummy 相対速度（使わない）
                    ], axis=1)  # (15, 12)
                except:
                    continue

                # 推論用に own[i+14] も保存しておく（最終フレームの自車速度）
                self.items.append((feat.astype(np.float32), own[i+14], sid, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, own_last, sid, frame_idx = self.items[idx]
        return torch.tensor(feat), own_last, sid, frame_idx

# -------- 推論処理 --------
def predict_and_save_submission(model_path, annot_root, distance_json_path, save_path="submission.json"):
    dataset = InferenceDataset(annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ExtendedLSTMWithAttention(input_size=12).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    results = {}
    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device)
            preds = model(feats).cpu().numpy()
            own_speeds = own_speeds.numpy()
            abs_speeds = preds + own_speeds  # TgtSpeed = Rel + Own

            for sid, frame_idx, tgt in zip(sids, frame_idxs, abs_speeds):
                if sid not in results:
                    results[sid] = []
                while len(results[sid]) <= frame_idx:
                    results[sid].append(None)  # padding
                results[sid][frame_idx] = float(round(tgt, 3))  # 小数第3位まで

    # 補完（欠損フレームを前の値で埋める）
    for sid, values in results.items():
        for i in range(len(values)):
            if values[i] is None:
                values[i] = values[i-1] if i > 0 else 0.0

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"✅ submission.json を {save_path} に保存しました")

# -------- 実行 --------
if __name__ == "__main__":
    predict_and_save_submission(
        model_path="model_extended_240d.pth",
        annot_root="../test/test_annotations",
        distance_json_path="../distance3/distance_estimates_test.json",
        save_path="submission.json"
    )


In [10]:
import os
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

# --- モデル定義 ---
class ExtendedLSTMWithAttention(nn.Module):
    def __init__(self, input_size=12, hidden_size=128):
        super().__init__()
        self.pre_fc = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=hidden_size,
                            num_layers=2, batch_first=True, dropout=0.3)
        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.dropout = nn.Dropout(0.3)
        self.out_fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, T, F = x.size()
        x = self.pre_fc(x.view(-1, F)).view(B, T, -1)
        lstm_out, _ = self.lstm(x)
        attn_scores = self.attn_fc(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.out_fc(self.dropout(context)).squeeze(1)

# --- 推論用データセット ---
class InferenceDataset(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        self.seq_lens = {}

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            self.seq_lens[sid] = len(seq)

            if len(seq) < 15 or sid not in self.distances:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            strdeg = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 15:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            own1 = np.gradient(own)
            own2 = np.gradient(own1)
            str1 = np.gradient(strdeg)
            dist1 = np.gradient(dist)
            dist2 = np.gradient(dist1)

            def smooth(x, w):
                if len(x) < w:
                    return np.zeros_like(x)
                return np.convolve(x, np.ones(w)/w, mode='same')

            sm3 = smooth(dist, 3)
            sm5 = smooth(dist, 5)
            sm7 = smooth(dist, 7)

            for i in range(len(seq) - 14):
                rel_speed = np.zeros(15)  # placeholder
                try:
                    feat = np.stack([
                        own[i:i+15], strdeg[i:i+15], dist[i:i+15],
                        own1[i:i+15], str1[i:i+15], dist1[i:i+15],
                        own2[i:i+15], dist2[i:i+15],
                        sm3[i:i+15], sm5[i:i+15], sm7[i:i+15],
                        rel_speed
                    ], axis=1)
                except:
                    continue

                self.items.append((feat.astype(np.float32), own[i+14], sid, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, own_last, sid, frame_idx = self.items[idx]
        return torch.tensor(feat), own_last, sid, frame_idx

# --- 推論と submission.json 出力 ---
def predict_and_save_submission(
    model_path,
    annot_root,
    distance_json_path,
    save_path="submission.json"
):
    dataset = InferenceDataset(annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ExtendedLSTMWithAttention(input_size=12).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    raw_preds = defaultdict(list)

    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device)
            preds = model(feats).cpu().numpy()
            own_speeds = own_speeds.numpy()
            abs_speeds = preds + own_speeds

            for sid, frame_idx, tgt in zip(sids, frame_idxs, abs_speeds):
                raw_preds[sid].append((frame_idx + 14, float(round(tgt, 3))))  # +14フレーム補正

    submission = {}
    for sid, pairs in raw_preds.items():
        pairs.sort()
        seq_len = dataset.seq_lens.get(sid, max(f for f, _ in pairs) + 1)
        pred_list = [0.0] * seq_len
        for idx, val in pairs:
            if idx < seq_len:
                pred_list[idx] = val
        for i in range(1, seq_len):
            if pred_list[i] == 0.0:
                pred_list[i] = pred_list[i-1]
        submission[sid] = pred_list

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    print(f"✅ 完成: {save_path} に保存されました（全シーン {len(submission)} 件）")

# --- 実行ブロック ---
if __name__ == "__main__":
    predict_and_save_submission(
        model_path="model_extended_240d.pth",
        annot_root="../test/test_annotations",
        distance_json_path="../testdistance/testdistance_estimates.json",
        save_path="submission.json"
    )


100%|██████████| 414/414 [00:01<00:00, 302.64it/s]


✅ 完成: submission.json に保存されました（全シーン 239 件）


In [13]:
import os
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

# --- モデル定義 ---
class ExtendedLSTMWithAttention(nn.Module):
    def __init__(self, input_size=12, hidden_size=128):
        super().__init__()
        self.pre_fc = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=32, hidden_size=hidden_size,
                            num_layers=2, batch_first=True, dropout=0.3)
        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.dropout = nn.Dropout(0.3)
        self.out_fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B, T, F = x.size()
        x = self.pre_fc(x.view(-1, F)).view(B, T, -1)
        lstm_out, _ = self.lstm(x)
        attn_scores = self.attn_fc(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.out_fc(self.dropout(context)).squeeze(1)

# --- 推論用データセット ---
class InferenceDataset(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        self.seq_lens = {}

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            self.seq_lens[sid] = len(seq)

            if len(seq) < 15 or sid not in self.distances:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            strdeg = np.array([f['StrDeg'] for f in seq], dtype=np.float32)

            keys = sorted(self.distances[sid].keys())
            if len(keys) < 15:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            own1 = np.gradient(own)
            own2 = np.gradient(own1)
            str1 = np.gradient(strdeg)
            dist1 = np.gradient(dist)
            dist2 = np.gradient(dist1)

            def smooth(x, w):
                if len(x) < w:
                    return np.zeros_like(x)
                return np.convolve(x, np.ones(w)/w, mode='same')

            sm3 = smooth(dist, 3)
            sm5 = smooth(dist, 5)
            sm7 = smooth(dist, 7)

            for i in range(len(seq) - 14):
                rel_speed = np.zeros(15)  # placeholder
                try:
                    feat = np.stack([
                        own[i:i+15], strdeg[i:i+15], dist[i:i+15],
                        own1[i:i+15], str1[i:i+15], dist1[i:i+15],
                        own2[i:i+15], dist2[i:i+15],
                        sm3[i:i+15], sm5[i:i+15], sm7[i:i+15],
                        rel_speed
                    ], axis=1)
                except:
                    continue

                own_avg = np.mean(own[i:i+15])  # ← 修正点
                self.items.append((feat.astype(np.float32), own_avg, sid, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, own_avg, sid, frame_idx = self.items[idx]
        return torch.tensor(feat), own_avg, sid, frame_idx

# --- 推論と submission.json 出力 ---
def predict_and_save_submission(
    model_path,
    annot_root,
    distance_json_path,
    save_path="submission.json"
):
    # 🔍 デバッグ出力
    print("🔎 推論準備：アノテーション・距離データ検査開始")
    print(f"📂 アノテーションフォルダ: {annot_root}")
    print(f"📄 距離ファイル: {distance_json_path}")

    all_jsons = [f for f in os.listdir(annot_root) if f.endswith(".json")]
    print(f"🗂 アノテーション件数: {len(all_jsons)} → {all_jsons[:3]} ...")

    with open(distance_json_path, encoding='utf-8') as f:
        distances = json.load(f)
    print(f"📏 距離データのscene数: {len(distances)} → {list(distances.keys())[:3]} ...")

    dataset = InferenceDataset(annot_root, distance_json_path)
    print(f"📦 推論対象Scene数（実データ内）: {len(set(i[2] for i in dataset.items))}")
    print(f"📊 推論対象サンプル数（15フレーム単位）: {len(dataset)}")

    if len(dataset) > 0:
        feat, own, sid, idx = dataset[0]
        print(f"✅ 例: sid={sid}, frame_idx={idx}, OwnSpeed(平均)={own:.3f}, feat_shape={feat.shape}")
    else:
        print("⚠️ 推論対象が0件です。距離データまたはアノテーション不足の可能性があります。")

    # 推論本体
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ExtendedLSTMWithAttention(input_size=12).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    raw_preds = defaultdict(list)
    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device)
            preds = model(feats).cpu().numpy()
            own_speeds = own_speeds.numpy()
            abs_speeds = preds + own_speeds  # ← OwnSpeed平均を加算

            for sid, frame_idx, tgt in zip(sids, frame_idxs, abs_speeds):
                raw_preds[sid].append((frame_idx + 14, float(round(tgt, 3))))  # +14補正

    # フル長に整形・補完
    submission = {}
    for sid, pairs in raw_preds.items():
        pairs.sort()
        seq_len = dataset.seq_lens.get(sid, max(f for f, _ in pairs) + 1)
        pred_list = [0.0] * seq_len
        for idx, val in pairs:
            if idx < seq_len:
                pred_list[idx] = val
        for i in range(1, seq_len):
            if pred_list[i] == 0.0:
                pred_list[i] = pred_list[i-1]
        submission[sid] = pred_list

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    print(f"✅ 完了: {save_path} に保存（scene数: {len(submission)}）")

# --- 実行 ---
if __name__ == "__main__":
    predict_and_save_submission(
        model_path="model_extended_240d.pth",
        annot_root="../test/test_annotations",
        distance_json_path="../testdistance/testdistance_estimates.json",
        save_path="submission.json"
    )


🔎 推論準備：アノテーション・距離データ検査開始
📂 アノテーションフォルダ: ../test/test_annotations
📄 距離ファイル: ../testdistance/testdistance_estimates.json
🗂 アノテーション件数: 239 → ['213.json', '016.json', '216.json'] ...
📏 距離データのscene数: 239 → ['000', '001', '002'] ...
📦 推論対象Scene数（実データ内）: 239
📊 推論対象サンプル数（15フレーム単位）: 26472
✅ 例: sid=000, frame_idx=0, OwnSpeed(平均)=40.320, feat_shape=torch.Size([15, 12])


100%|██████████| 414/414 [00:01<00:00, 262.01it/s]


✅ 完了: submission.json に保存（scene数: 239）
